# **<font color='#6edce9'>Validacion de consolidado</font>**
### **Objetivo:** Construir el pipeline de mapeo y limpieza para unificar los datasets anuales de RRHH y migrar la base de datos consolidada a PostgreSQL.

## **<font color='#eda985'> 1. Importación y llamada a función para consolidar los 7 excel (2019 hasta 2025)</font>**

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))  # sube un nivel desde notebooks/ hacia la raíz

from src.minsa_analystics.etl import consolidar_datos

# ============================================================
# CONFIGURACIÓN
# ============================================================
# --- Carga con caché en Parquet, para no re-procesar los 7 Excel cada vez ---
FORZAR_RECARGA = False # IMPORTANTE⚠️: . cambia a True cuando se modifique algo  etl.py o corrijas datos

ruta_datos = str(Path.cwd().parent / 'data' / 'raw')
ruta_parquet = Path.cwd().parent / 'data' / 'processed' / 'consolidado.parquet'

# ============================================================
# CARGA / GENERACIÓN DEL CONSOLIDADO
# ============================================================
if ruta_parquet.exists() and not FORZAR_RECARGA:
    print("Cargando consolidado desde cache (parquet)...")
    df = pd.read_parquet(ruta_parquet)
else:
    print("Generando consolidado desde los 7 Excel...")
    df = consolidar_datos(str(ruta_datos))

    # Crear carpeta si no existe
    ruta_parquet.parent.mkdir(parents=True, exist_ok=True)

    # Guardar Parquet
    df.to_parquet(ruta_parquet, index=False)

    print(f"\nParquet guardado en:")
    print(ruta_parquet)
    print(f"\n¿Archivo creado?: {ruta_parquet.exists()}")

    if ruta_parquet.exists():
        print(f"Tamaño: {ruta_parquet.stat().st_size / 1024**2:.2f} MB")


print(f"\nDimensiones: {df.shape}")

In [ ]:
# Verificando datos del campo 'es_especialista'
print(df.groupby(['anio_registro', 'es_especialista'], dropna=False).size().unstack(fill_value=0))


## **<font color='#eda985'> 2. Verificando cantidad de registros</font>**

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)

# Filas por año (deberían coincidir con lo que ya viste arriba)
print('Cantidad de registros por año')
print('='*30)
print(df['anio_registro'].value_counts().sort_index())

In [ ]:
df.head(5)

In [ ]:
df.columns

In [ ]:
# Revisando tipo de dato de cada columna

# for col in df.columns:
#     tipos = df[col].map(type).unique()
#     if len(tipos) > 1:
#         print(col, '->', tipos)

for col in df.columns:
    tipos = df[col].map(type).unique()
    print(f'{col:<28} -> {tipos}')

## **<font color='#eda985'> 3. Verificando nulos de cada columna y filas con espacios al inicio o final</font>**

In [ ]:
# % de nulos por columna, para detectar algo inesperado
print('% Columnas Nulas')
print('='*30)
print(df.isna().mean().sort_values(ascending=False).head(10))

# Resultado inicial primer parquet
''' 
% Columnas Nulas
==============================
id_especialidad          0.7159
especialidad             0.7159
condicion_especialidad   0.7156
microrred                0.1153
red                      0.0835
edad                     0.0021
sexo                     0.0003
clasificacion            0.0003
distrito                 0.0000
provincia                0.0000
dtype: float64
'''

In [ ]:
# Verificando filas con espacios al inicio o al final
cols_texto = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
for col in cols_texto:
    s = df[col].dropna()
    n = (s != s.str.strip()).sum()
    if n:
        print(f'{col}: {n} filas con espacios al inicio o al final')

# Resultado inicial primer parquet
'''
departamento: 7 filas con espacios al inicio o al final
provincia: 7 filas con espacios al inicio o al final
distrito: 52 filas con espacios al inicio o al final
clasificacion: 3008 filas con espacios al inicio o al final
descripcionestablecimiento: 3895 filas con espacios al inicio o al final
categoria: 42 filas con espacios al inicio o al final
condicion_laboral: 311657 filas con espacios al inicio o al final
'''

In [ ]:
# Verificando filas con dobles espacios

cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
for col in cols:
    n = df[col].str.contains(r'\s{2,}', na=False).sum()
    if n:
        antes = df[col].nunique()
        despues = df[col].str.replace(r'\s+', ' ', regex=True).nunique()
        print(f'{col:<27}: {n} filas con dobles espacios | únicos {antes} -> {despues}')


# Resultado inicial primer parquet
''' 
uedescripue                : 9239 filas con dobles espacios | únicos 238 -> 238
red                        : 1058 filas con dobles espacios | únicos 161 -> 161
microrred                  : 325 filas con dobles espacios | únicos 882 -> 882
clasificacion              : 201 filas con dobles espacios | únicos 55 -> 55
descripcionestablecimiento : 63871 filas con dobles espacios | únicos 9802 -> 9780
especialidad               : 2579 filas con dobles espacios | únicos 356 -> 353
'''

In [ ]:
# Verificando registros con tildes
patron = r'[ÁÉÍÓÚÜáéíóúü]'
for col in [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]:
    n = df[col].str.contains(patron, na=False).sum()
    if n:
        print(f'{col:<15}: {n} filas con tildes')


# Resultado inicial primer parquet
''' 
uedescripue: 6871 filas con tildes
provincia: 52 filas con tildes
distrito: 84 filas con tildes
clasificacion: 257 filas con tildes
descripcionestablecimiento: 26981 filas con tildes
categoria: 9967 filas con tildes
id_especialidad: 1 filas con tildes
especialidad: 116879 filas con tildes
regimen_laboral: 131867 filas con tildes
condicion_laboral: 131823 filas con tildes
grupofinal3: 427669 filas con tildes
'''

In [ ]:
# compara cuántos valores distintos tiene cada columna antes y después de limpiar las tildes
TILDES = str.maketrans('ÁÉÍÓÚÜáéíóúü', 'AEIOUUaeiouu')
cols = ['uedescripue', 'provincia', 'distrito', 'descripcionestablecimiento',
        'categoria', 'id_especialidad', 'especialidad', 'regimen_laboral',
        'condicion_laboral', 'grupofinal3']

for col in cols:
    antes = df[col].nunique()
    despues = df[col].str.strip().str.translate(TILDES).nunique()
    print(f'{col:<27}: {antes} -> {despues}')

    #print(f'{col:<15}: {n} filas con tildes')

# Resultado inicial primer parquet
''' 
uedescripue: 240 -> 240
provincia: 198 -> 196
distrito: 1737 -> 1734
clasificacion: 68 -> 55
descripcionestablecimiento: 9820 -> 9802
categoria: 27 -> 25
id_especialidad: 395 -> 395
especialidad: 495 -> 356
regimen_laboral: 12 -> 11
condicion_laboral: 72 -> 58
grupofinal3: 21 -> 21
'''

In [ ]:
# Distribución de las tildes en las columnas grandes
patron = r'[ÁÉÍÓÚÜáéíóúü]'
col = 'grupofinal3'
con_tilde = df[col].str.contains(patron, na=False)
print(df.loc[con_tilde].groupby('anio_registro').size())
print(df.loc[con_tilde, col].value_counts().head(10))

# Resultado inicial primer parquet
''' 
anio_registro
2019    44618
2020    55690
2021    69674
2022    61329
2023    71310
2024    61522
2025    63526
dtype: int64
grupofinal3
Médico                     257387
Odontólogo                  41847
Psicólogo                   38715
Tecnólogo Médico            31916
Químico Farmacéutico        28383
Biólogo                     24436
Médico Veterinario           2542
Técnicos Especializados      1901
Químico                       542
Name: count, dtype: int64
'''

## **<font color='#eda985'> 4. Verificando cada columna</font>**

In [ ]:
# ======== VERIFICACION CAMPO POR CAMPO ==============
def revisar(df, col, n=15):
    print(f'Tipo de dato: {df[col].dtype}')
    print('======= NULOS POR AÑO =======')
    print(df.groupby('anio_registro')[col].apply(lambda s: s.isna().sum()).to_frame('nulos'))
    print('======= ÚNICOS (TOTAL) =======')
    print(df[col].nunique(dropna=False))
    print('======= ÚNICOS POR AÑO =======')
    print(df.groupby('anio_registro')[col].nunique(dropna=False).to_frame('unicos'))
    vc = df[col].value_counts(dropna=False)
    print(f'======= {n} MÁS FRECUENTES =======')
    print(vc.head(n))
    print(f'======= {n} MENOS FRECUENTES =======')
    print(vc.tail(n))
    print('======= TIPOS MEZCLADOS POR AÑO =======')
    print(df.groupby('anio_registro')[col].agg(lambda s: s.map(type).nunique()))


### **<font color='#96f499'> 4.1. Verificando columna 'pea'  </font>**

In [ ]:
#revisar(df, 'pea') # Constante = 1 

### **<font color='#96f499'> 4.2. Verificando columna 'renaes'  </font>**

In [ ]:
revisar(df, 'renaes')

### **<font color='#96f499'> 4.3. Verificando columna 'pliegodescrip'  </font>**

In [ ]:
revisar(df, 'pliegodescrip')

In [ ]:
col = 'pliegodescrip' 
df[col].value_counts(dropna=False).sort_values()

### **<font color='#96f499'> 4.4. Verificando columna 'uedescripue'  </font>**

In [ ]:
revisar(df, 'uedescripue')

In [ ]:
col = 'uedescripue' 
df[col].value_counts(dropna=False).sort_values()

In [ ]:
# ==== VERIFICANDO NOMBRES POR CODIGO =====

cod = df['uedescripue'].str.extract(r'^(\d+)')[0] # dividimos el texto extrayendo los digitos del texto
print(f'CODIGOS UNICOS DE UE:{cod.nunique()}')

print('\nCODIGOS CON MAS DE UNA DESCRIPCION')
nombres_por_codigo = df.assign(cod=cod).groupby('cod')['uedescripue'].nunique()
print(nombres_por_codigo[nombres_por_codigo > 1])   # códigos con más de un nombre

In [ ]:
# Como se ve que los dos codigo '0901' y '1712' tienen 2 descripciones diferetes evaluamos en que excel
for c in ['0901', '1712']:
    print(f'\n=== Código {c} ===')
    print(
        df.loc[cod == c]
        .groupby(['uedescripue', 'anio_registro']).size()
        .unstack(fill_value=0)
    )

### **<font color='#96f499'> 4.5. Verificando columna 'ubigeo'  </font>**

In [ ]:
revisar(df, 'ubigeo')

In [ ]:
# ========== VERIFICANDO LONGITUD DE DIGITOS = 6 ===========
print(df['ubigeo'].str.len().value_counts())          
print((~df['ubigeo'].str.fullmatch(r'\d{6}')).sum())  # debe dar 0

### **<font color='#96f499'> 4.6. Verificando columna 'departamento'  </font>**

In [ ]:
revisar(df, 'departamento')

In [ ]:
col = 'departamento' 
df[col].value_counts(dropna=False).sort_values()

In [ ]:
cruce = df.assign(cod_dep=df['ubigeo'].str[:2]).groupby('cod_dep')['departamento'].nunique()
print(cruce[cruce > 1])   # códigos de departamento con más de un nombre

In [ ]:
cod_dep = df['ubigeo'].str[:2]
print(
    df.loc[cod_dep == '13'].groupby('departamento').size()
)

In [ ]:
cod_dep = df['ubigeo'].str[:2]
print(
    df.loc[cod_dep == '13']
    .groupby(['departamento', 'anio_registro']).size()
    .unstack(fill_value=0)
)

cod_dep = df['ubigeo'].str[:2]

In [ ]:
print(df.loc[cod_dep == '13', 'departamento'].map(repr).value_counts())

### **<font color='#96f499'> 4.7. Verificando columna 'provincia'  </font>**

In [ ]:
revisar(df, 'provincia')

In [ ]:
cruce = df.assign(cod_prov=df['ubigeo'].str[:4]).groupby('cod_prov')['provincia'].nunique()
print(cruce[cruce > 1])   # códigos de provincia con más de un nombre

In [ ]:
cod_prov = df['ubigeo'].str[:4]
print(
    df.loc[cod_prov == '0809'].groupby('provincia').size()
)


# Resultado inicial primer parquet
''' 
provincia
LA CONVENCION    12992
LA CONVENCIÓN       52
dtype: int64
'''

In [ ]:
cod_prov = df['ubigeo'].str[:4]
print(
    df.loc[cod_prov == '0809'].groupby(['provincia', 'anio_registro']).size().unstack(fill_value=0)
)

cod_prov = df['ubigeo'].str[:4]

# Resultado inicial primer parquet
'''
anio_registro  2019  2020  2021  2022  2023  2024  2025
provincia                                              
LA CONVENCION  1480  1637  1854  1899  1951  1996  2175
LA CONVENCIÓN     0     0     0     0     0    52     0 
'''


### **<font color='#96f499'> 4.8. Verificando columna 'distrito'  </font>**

In [ ]:
revisar(df, 'distrito')

In [ ]:
# Un solo nombre de distrito por cada ubigeo (6 dígitos)
cruce = df.groupby('ubigeo')['distrito'].nunique()
print(cruce[cruce > 1])

# Resultado inicial primer parquet
''' 
ubigeo
130112    2
Name: distrito, dtype: int64
'''


In [ ]:
cod_distrito = df['ubigeo']
print(
    df.loc[cod_distrito == '130112'].groupby('distrito').size()
)


# Resultado inicial primer parquet
''' 
distrito
ALTO TRUJILLO    130
VIR                7
dtype: int64
'''

In [ ]:
cod_distrito = df['ubigeo']
print(
    df.loc[cod_distrito == '130112'].groupby(['distrito', 'anio_registro']).size().unstack(fill_value=0)
)



# Resultado inicial primer parquet
'''
anio_registro  2024  2025
distrito                 
ALTO TRUJILLO     0   130
VIR               7     0 
'''


In [ ]:
print(df.loc[(df['ubigeo'] == '130112') & (df['distrito'] == 'VIR'),
             ['anio_registro', 'provincia', 'renaes', 'descripcionestablecimiento']])

# Resultado inicial primer parquet
''' 
anio_registro provincia    renaes descripcionestablecimiento
1514016           2024  TRUJILLO  00012229                    BARRIO1
1629240           2024  TRUJILLO  00005220             ALTO  TRUJILLO
1632038           2024  TRUJILLO  00005220             ALTO  TRUJILLO
1632049           2024  TRUJILLO  00005220             ALTO  TRUJILLO
1632067           2024  TRUJILLO  00005220             ALTO  TRUJILLO
1632233           2024  TRUJILLO  00012229                    BARRIO1
1632366           2024  TRUJILLO  00005220             ALTO  TRUJILLO
'''

In [ ]:
# COMPROBACION UBICACION GEOGRAFICA (departamento,, provincia, distrito)
chequeo = df.groupby('ubigeo')[['departamento', 'provincia', 'distrito']].nunique()
print(chequeo[(chequeo > 1).any(axis=1)])

# Resultado inicial primer parquet
''' 
        departamento  provincia  distrito
ubigeo                                   
130112             1          1         2
'''

### **<font color='#96f499'> 4.9. Verificando columna 'diresa'  </font>**

In [ ]:
revisar(df, 'diresa')

In [ ]:
col = 'diresa' 
df[col].value_counts(dropna=False).sort_values()

### **<font color='#96f499'> 4.10. Verificando columna 'red'  </font>**

In [ ]:
revisar(df, 'red')

In [ ]:
col = 'red' 
df[col].value_counts(dropna=False).sort_values()


In [ ]:
col = "red"
s = df[col]
contiene = s.str.contains("a ninguna", case=False, na=False)

print(f"'-' total: {(s == '-').sum():,}")
print(f"NaN total: {s.isna().sum():,}")
print(f"Contienen 'a ninguna': {contiene.sum():,}")
print(s[contiene].value_counts())   # cada variante con su conteo

# Resultado inicial primer parquet
''' 
'-' total: 20,469
NaN total: 160,360
Contienen 'a ninguna': 775,409
red
NO PERTENECE A NINGUNA RED    775409
Name: count, dtype: int64
'''


### **<font color='#96f499'> 4.11. Verificando columna 'microrred'  </font>**

In [ ]:
revisar(df, 'microrred')

In [ ]:
col = 'microrred' 
df[col].value_counts(dropna=False).sort_values()

In [ ]:
col = "microrred"
s = df[col]
contiene = s.str.contains("a ninguna", case=False, na=False)

print(f"'-' total: {(s == '-').sum():,}")
print(f"NaN total: {s.isna().sum():,}")
print(f"Contienen 'a ninguna': {contiene.sum():,}")
print(s[contiene].value_counts())   # cada variante con su conteo

# Resultado inicial primer parquet
''' 
'-' total: 33,527
NaN total: 221,397
Contienen 'a ninguna': 977,364
microrred
NO PERTENECE A NINGUNA MICRORED     975516
NO PERTENECE A NINGUNA MICRORRED      1700
NO PERNTENE A NINGUNA MICRORED         148
Name: count, dtype: int64
'''

### **<font color='#96f499'> 4.12. Verificando columna 'clasificacion'  </font>**

In [ ]:
#revisar(df, 'clasificacion')

In [ ]:
#col = 'clasificacion' 
#df[col].value_counts(dropna=False).sort_values()

### **<font color='#96f499'> 4.13. Verificando columna 'descripcionestablecimiento'  </font>**

In [ ]:
revisar(df, 'descripcionestablecimiento')

In [ ]:
col = 'descripcionestablecimiento' 
df[col].value_counts(dropna=False).sort_values()

In [ ]:
d = 'descripcionestablecimiento'
n = df.groupby('renaes')[d].nunique()
multi = n[n > 1]
print(f"renaes únicos: {df['renaes'].nunique():,}")
print(f"renaes con >1 nombre: {len(multi):,} ({df['renaes'].isin(multi.index).sum():,} filas)")
print('----------------------------------------------------------------------------------')

# detalle de los casos con más nombres: cada nombre con filas y rango de años
for r in multi.sort_values(ascending=False).head(2).index:
    print(f'\nRENAES {r}')
    print(df[df['renaes'] == r].groupby(d)['anio_registro'].agg(['count', 'min', 'max']))
print('----------------------------------------------------------------------------------')

# inverso: nombres repetidos en más de un renaes
m = df.groupby(d)['renaes'].nunique()
print(f"\nnombres con >1 renaes: {(m > 1).sum():,}")

# Resultado inicial primer parquet
''' 
renaes únicos: 9,056
renaes con >1 nombre: 1,856 (558,603 filas)
----------------------------------------------------------------------------------

RENAES 00007326
                            count   min   max
descripcionestablecimiento                   
C.S. I-3 INDEPENDENCIA        194  2023  2025
P.S. I-2 INDEPENDENCIA         64  2020  2021
P.S. I-3 INDEPENDENCIA         44  2022  2022
P.S. INDEPENDENCIA             35  2019  2019

RENAES 00006580
                                                    count   min   max
descripcionestablecimiento                                           
HOSPITAL DE TOCACHE                                   138  2019  2019
HOSPITAL II-1 "DR. JOSE PEÑA PORTUGUEZ " - TOCACHE   1059  2023  2025
HOSPITAL II-1 "DR. JOSE PEÑA PORTUGUEZ" - TOCACHE     517  2024  2024
HOSPITAL II-1 TOCACHE                                1648  2020  2022
----------------------------------------------------------------------------------

nombres con >1 renaes: 650
'''


### **<font color='#96f499'> 4.14. Verificando columna 'categoria'  </font>**

In [ ]:
revisar(df, 'categoria')

In [ ]:

df['categoria'].value_counts(dropna=False).sort_values()

In [ ]:
# Cuantos establecimientos cambiaron
cat = df['categoria'].str.upper()   # une 'Sin Categoría' / 'Sin categoría'
n = df.assign(cat=cat).groupby('renaes')['cat'].nunique()
renaes_cambio = n[n > 1].index
print(f"renaes con cambio de categoría: {len(renaes_cambio):,} de {len(n):,} ({len(renaes_cambio)/len(n):.1%})")


In [ ]:
# detalle por establecimiento, con su categoría año a año
cambio = df[df['renaes'].isin(renaes_cambio)].assign(cat=cat)

nombre = (cambio.sort_values('anio_registro')
                .drop_duplicates('renaes', keep='last')
                .set_index('renaes')['descripcionestablecimiento'])

detalle = (cambio.pivot_table(index='renaes', columns='anio_registro',
                              values='cat', aggfunc='first')
                 .assign(establecimiento=nombre))
detalle.insert(0, 'establecimiento', detalle.pop('establecimiento'))
detalle

In [ ]:
df.loc[df['flag_cambio_categoria'], 'renaes'].nunique()

### **<font color='#96f499'> 4.15. Verificando columna 'quintil'  </font>**

In [ ]:
revisar(df, 'quintil')

In [ ]:
# Verificando quintil del distrito de Miraflores
miraflores = df[
    (df['departamento'] == 'LIMA')
    & (df['provincia'] == 'LIMA')
    & (df['distrito'] == 'MIRAFLORES')
]
print(miraflores['ubigeo'].unique())
print(miraflores.groupby(['ubigeo', 'quintil']).size())
miraflores[['renaes', 'descripcionestablecimiento', 'ubigeo', 'distrito', 'quintil']].drop_duplicates('renaes').head()

In [ ]:
# Evaluando Ubigeos con mas de un(1) quintil

n= df.groupby('ubigeo')['quintil'].nunique()
ubigeos = n[n>1]
print(f"ubigeo con más de un quintil: {len(ubigeos)} de {len(n):,}")
print(ubigeos)

# Resultado inicial primer parquet
''' 
ubigeo con más de un quintil: 1 de 1,888
ubigeo
130112    2
'''


In [ ]:
(df[df['ubigeo'].isin(ubigeos.index)]
   .groupby(['ubigeo', 'departamento', 'provincia', 'distrito', 'anio_registro', 'quintil'])
   .agg(filas=('renaes', 'size'), establecimientos=('renaes', 'nunique'))
   .reset_index())

# Resultado inicial primer parquet --> En la siguiente celda

In [ ]:
print((df[df['ubigeo'].isin(ubigeos.index)]
   .groupby(['ubigeo', 'departamento', 'provincia', 'distrito', 'anio_registro', 'quintil'])
   .agg(filas=('renaes', 'size'), establecimientos=('renaes', 'nunique'))
   .reset_index()))

# Resultado inicial primer parquet
''' 
ubigeo departamento provincia       distrito       anio_registro  quintil     filas  establecimientos
0  130112  LA LIBERTAD  TRUJILLO  ALTO TRUJILLO           2024        2        7                 2 
1  130112  LA LIBERTAD  TRUJILLO  ALTO TRUJILLO           2025        3      130                 2 
'''

In [ ]:
# Veriricando los 7 registros
afect = df.loc[(df['ubigeo'] == '130112') & (df['anio_registro'] == 2024), 'renaes'].unique()
x = df[df['renaes'].isin(afect)]

# filas por establecimiento y año
print(x.groupby(['renaes', 'anio_registro']).size().unstack(fill_value=0))
print('-'*80)


# ubigeo, distrito y quintil que tuvo cada año
print(x.groupby(['renaes', 'anio_registro', 'ubigeo', 'distrito', 'quintil']).size())

### **<font color='#96f499'> 4.1. Verificando columna 'pcm'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'distfrontera'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'zaf2014final'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'estrategicos'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'sexo'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'es_especialista'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'id_especialidad'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'especialidad'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'condicion_especialidad'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'regimen_laboral'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'condicion_laboral'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'id_cargo'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'cargo'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'grupofinal2'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'grupofinal3'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'edad'  </font>**

### **<font color='#f496cd'> 4.1. Registro de validación del consolidado  </font>**

## Registro de validación del consolidado (hasta `quintil`)

### Reglas transversales (aplicadas en `limpiar_texto`)

| Regla | Qué hace | Columnas |
|---|---|---|
| Espacios | Recorta espacios al inicio y al final y colapsa espacios dobles | Todas las columnas de texto |
| Tildes | Quita tildes de las vocales (conserva la Ñ) | Las de `COLUMNAS_SIN_TILDES`: `provincia`, `descripcionestablecimiento`, `especialidad`, `regimen_laboral`, `condicion_laboral` |
| Mayúsculas | Pasa todo a mayúscula | `diresa`, `red`, `microrred`, `categoria` |

### Decisiones por columna

| Columna | Hallazgo | Decisión | Dónde se aplica |
|---|---|---|---|
| `pea` | Constante igual a 1 (contador de tablas dinámicas) | **Descartar** | `eliminar_columnas_innecesarias` |
| `renaes` | Texto de 8 caracteres, sin nulos, 9,056 códigos. Sin problemas | Conservar; es la **clave del establecimiento** | — |
| `pliegodescrip` | 30 valores, sin nulos ni variantes | Conservar | — |
| `uedescripue` | 238 códigos y 240 nombres: el `0901` cambió de categoría (III → II-2 en 2024) y el `1712` corrigió su nombre en 2022 | Conservar; unificar al nombre más reciente por código (queda 238 códigos y 238 nombres) | `unificar_unidades_ejecutoras` |
| `ubigeo` | Texto, sin nulos, 1,888 valores, siempre 6 dígitos con ceros a la izquierda. Algunos establecimientos cambian de `ubigeo` entre años (reasignación de distrito) | Conservar; es la **clave del distrito** | — |
| `departamento` | 26 valores (esperados 25): `LA LIBERTAD` duplicada por espacios (7 filas en 2024) | Conservar | `limpiar_texto` |
| `provincia` | 197 valores (esperados 196): `LA CONVENCIÓN` con tilde (52 filas, solo 2024) frente a `LA CONVENCION`. Espacios sobrantes en 7 filas | Conservar; quitar tildes → 196 valores | `limpiar_texto` |
| `distrito` | 1,734 valores, sin nulos ni tipos mezclados. Espacios sobrantes en 52 filas y 84 filas con tildes sin duplicados. El `ubigeo` `130112` tenía dos nombres: `ALTO TRUJILLO` (130 filas, 2025) y `VIR` (7 filas, 2024; son de Alto Trujillo, provincia Trujillo) | Conservar; corregir `VIR` → `ALTO TRUJILLO` con regla explícita | `limpiar_texto`, `corregir_distritos` |
| `diresa` | 31 valores (30 en 2019-2022). `SEGURO INTEGRAL DE SALUD` aparece como DIRESA solo desde 2023 (antes estaba en `MINSA ADMINISTRATIVO`). `LIMA` es el Gobierno Regional de Lima, no Lima Metropolitana | Conservar; **no usar como dimensión geográfica** (el pliego ya la contiene); documentar el quiebre de 2023 | `limpiar_texto` |
| `red` | Sin dato en tres formas: `NaN` (160,360), `'-'` (20,469, solo 2025, donde antes venía vacío) y `NO PERTENECE A NINGUNA RED` (775,409) | Nulos, `'-'` y variantes de "no pertenece" → `NO PERTENECE A NINGUNA RED` | `unificar_sin_red` |
| `microrred` | Igual que `red` (`'-'` en 33,527 filas de 2025). Tres grafías de "no pertenece": `MICRORED` (975,516), `MICRORRED` (1,700) y el tipeo `PERNTENE` (148). 74,095 filas tienen red real pero microrred vacía | Todo → `NO PERTENECE A NINGUNA MICRORED` (queda 0 nulos y 0 `'-'`) | `unificar_sin_red` |
| `clasificacion` | 55 valores que mezclan tipos de establecimiento con unidades administrativas (RED, DIRESA, MINSA, DIRIS…). Sin cambios por `renaes` entre años. Campo sin actualizar y no usado en reportes | **Descartar** | `eliminar_columnas_innecesarias` |
| `descripcionestablecimiento` | 9,780 nombres para 9,056 `renaes`. 1,856 `renaes` (558,603 filas) con más de un nombre por cambios de convención entre años, cambios de nivel y algún tipeo. 650 nombres se repiten en varios `renaes` (homónimos). 93 nombres con comillas | Conservar; quitar tildes y comillas; unificar al nombre más reciente por `renaes`. El nombre no es clave, `renaes` sí | `limpiar_texto`, `unificar_establecimientos` |
| `categoria` | 25 valores; `Sin Categoría` / `Sin categoría` duplicadas por mayúscula (6 filas en 2023). 657 `renaes` (127,193 filas) cambian de categoría entre años; predominan las subidas `I-1 → I-2` e `I-2 → I-3` | Conservar por año; pasar a mayúscula (24 valores); agregar `categoria_actual` (más reciente por `renaes`) y `flag_cambio_categoria` | `limpiar_texto`, `agregar_categoria_actual` |
| `quintil` | `int64`, sin nulos, valores 1 a 5 (1 = nivel más bajo, 5 = más alto; confirmado con Miraflores, Lima = 5). Atributo del distrito, constante en los 7 años para 1,887 de 1,888 `ubigeo`. En `130112` las 7 filas de 2024 traen 2 y en 2025 es 3. Esos dos establecimientos (`00005220` y `00012229`) tienen casi vacío 2024: 5 y 2 filas frente a 110-130 en otros años | Conservar; corregir las 7 filas a 3 con regla explícita; documentar la laguna de 2024 como limitación | `corregir_quintil` |

### Orden de las transformaciones en `consolidar_datos`

`normalizar_tipos` → `limpiar_texto` → `unificar_sin_red` → `corregir_distritos` → `corregir_quintil` → `unificar_unidades_ejecutoras` → `unificar_establecimientos` → `agregar_categoria_actual` → `aplicar_reglas_negocio` → `eliminar_columnas_innecesarias`

### Pendientes de validar

`pcm`, `distfrontera`, `zaf2014final`, `estrategicos`, `sexo`, `es_especialista` y columnas de especialidad, `regimen_laboral`, `condicion_laboral`, `id_cargo`, `cargo`, `grupofinal2`, `grupofinal3`, `edad`, `anio_registro` y los flags.

## **<font color='#eda985'> 5. Auditoría y Perfilado de Errores</font>**

### **<font color='#96f499'> 4.1. Verificando y corrigiendo campos Relacionados: id_especialidad, especialidad,condicion_especialidad, es_especialista </font>**

### Reglas de Negocio: Validación de Especialistas
Para validar la consistencia de los datos, la columna `es_especialista` debe cruzarse obligatoriamente con los campos de especialidad (`id_especialidad`, `especialidad`, `condicion_especialidad`) y `la condición laboral`.

##### 1. Cuando `es_especialista` es **"SI"** 🟢
**Regla estricta:** Todos los campos de especialidad deben estar **completamente llenos**.
**Campos obligatorios:**
  * `id_especialidad`
  * `especialidad`
  * `condicion_especialidad`
*Cualquier registro en "SI" con alguno de estos campos vacío se considera un **Error de Consistencia**.*

##### 2. Cuando `es_especialista` es **"NO"** 🟡
* **Regla flexible:** Los campos de especialidad **pueden o no estar llenos** (se permite que tengan datos o que estén vacíos).
* **Excepción por Residencia:** Si el registro tiene `condicion_laboral` = **"residente"**, el valor de `es_especialista` será obligatoriamente **"NO"**, incluso si ya tiene asignada una especialidad en los campos de texto.

---

> 💡 **Nota de Diagnóstico:** Tras analizar el set de datos, los **11 registros** con valor "SI" y especialidad vacía (`False`) representan la inconsistencia más grave, ya que violan la regla estricta del negocio. Los registros en "NO", "No" o "0" que contienen datos pueden justificarse bajo la excepción de médicos residentes.
Usa el código con precaución.
🛠️ Código de Pandas actualizado con la nueva regla
Para que tu script refleje exactamente esta nueva consideración de los residentes, puedes usar este enfoque para aislar únicamente los errores reales (los 11 "SI" vacíos y cualquier "SI" que sea residente por error):
python

In [ ]:
# 1. Filtro estricto: Es "SI" pero le falta alguna de las 3 columnas de especialidad
campos_incompletos = df['id_especialidad'].isna() | df['especialidad'].isna() | df['condicion_especialidad'].isna()
error_si_incompleto = (df['es_especialista'] == 'SI') & campos_incompletos

# 2. Filtro de contradicción: Dice "SI" pero su condición laboral es "residente" (debería ser NO)
error_residente_especialista = (df['es_especialista'] == 'SI') & (df['condicion_laboral'].str.lower() == 'residente')

# Unimos los errores reales del sistema
errores_reales = df[error_si_incompleto | error_residente_especialista]

print(f"Total de registros con errores críticos detectados: {len(errores_reales)}")
print(f'Total regitros indican SI es especialista pero alguno de los 3 campos vacios: {error_si_incompleto.sum()}')
print(f'Total regitros indican SI es especialista pero condicion laboral es Residente: {error_residente_especialista.sum()} ')

In [ ]:
# Verificando campo 'tiene_especialidad' con 'especialidad' NO VACÍO
df['tiene_especialidad'] = df['especialidad'].notna() # es_especialista vs especialidad llena
print(df.groupby(['es_especialista', 'tiene_especialidad'], dropna=False).size())

### **<font color='#96f499'> 4.2. Verificando detalle del campo EDAD </font>**

In [ ]:
# EDAD: rango razonable en todo el consolidado
print('Descripcion EDAD')
print('='*30)
print(df['edad'].describe())

In [ ]:
# Registros con edad mayo a 80 años
df[df['edad'] > 80][['edad', 'anio_registro']].value_counts()